# Data Cleaning Pipeline — Hyderabad House Data

**Input:** `datasets/housedata_updated.csv` (93,621 rows)  →  **Output:** `datasets/housedata_clean.csv` (62,047 rows, model-ready)

**Rule of the pipeline:** repair or impute whenever possible — drop a row only when it is a duplicate or its price (the target) is unusable.

### Pipeline steps

| Step | What happens |
|---|---|
| 1 | Remove duplicate rows (exact copies + same listing under a different URL) |
| 2 | Remove rows with unusable prices (rent listings, zero/missing, corrupt) |
| 3 | Fix messy text — one clean format per column |
| 4 | Impossible values (30-crore sqft, age 2031...) → set to missing |
| 5 | Contradicting fields → repair what we can, flag the rest |
| 6 | Recompute derived columns (PricePerSqft, Price_Cr, Age_Group) |
| 6b | Save `housedata_clean_without_imputation.csv` — cleaned, gaps left empty |
| 7 | Fill all missing values (MICE imputation — tested against alternatives) |
| 8 | Final guards (flag absurd price/area pairs, last dedupe) and save |

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('datasets/housedata_updated.csv', low_memory=False)
start_rows = len(df)
print('Loaded:', df.shape)

Loaded: (93621, 31)


## Step 1 — Remove duplicate rows

Two passes: exact copies of a row, and rows identical in every feature but the URL/address text (the same listing scraped twice, e.g. with a `#explore-nearby` link).

**Why:** copies add nothing, over-weight some listings, and leak between train/test splits.

*Not removed:* the same property listed in different months — that is real price movement over time, not duplication.

In [2]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} exact duplicates -> {len(df)} rows')

# same listing scraped twice under a slightly different URL (#fragment) or address text
feature_cols = [c for c in df.columns if c not in ('URL', 'Full_Address')]
before = len(df)
df = df.drop_duplicates(subset=feature_cols).reset_index(drop=True)
print(f'Removed {before - len(df)} near-duplicates (identical except URL/address) -> {len(df)} rows')

Removed 7069 exact duplicates -> 86552 rows
Removed 20074 near-duplicates (identical except URL/address) -> 66478 rows


## Step 2 — Keep only usable sale prices

Drop rows where the price is missing, zero, a rent amount that leaked into this sale data (₹10,000 "prices" from rent URLs), or corrupt (> ₹100 Cr).

**Why:** price is the prediction target — it cannot be imputed. Guessing the target teaches the model our guesses.

*Alternative:* keeping rent rows as a separate segment — rejected, no reliable rent/sale label for rows without a URL.

In [3]:
before = len(df)
is_rent = df['URL'].str.contains('for-rent|/rent', na=False) | (df['Price_INR'] < 200000)
bad_price = df['Price_INR'].isna() | (df['Price_INR'] <= 0) | (df['Price_INR'] > 1e10)
df = df[~(is_rent | bad_price)].reset_index(drop=True)
print(f'Removed {before - len(df)} rent/broken-price rows -> {len(df)} rows')

Removed 3402 rent/broken-price rows -> 63076 rows


## Step 3 — Fix messy text (no rows lost)

One clean format per column:
- `BHK`: "3 Bedrooms" → 3
- `Floor_Number`: "4th of 5 Floors" → 4 (and the 5 fills `Total_Floors` if missing); "Ground" → 0
- `Car_Parking`: "1 Covered, 1 Open" → 2
- `Furnishing / Construction_Status / Facing`: spelling and case variants merged
- Values that belong to a different column (addresses, "0.53 KM", "Ownership Type") → missing

**Why missing instead of dropping:** one corrupt cell shouldn't cost the 29 good cells in the same row.

In [4]:
# BHK -> numeric
df['BHK'] = pd.to_numeric(df['BHK'].astype('string').str.extract(r'(\d+)')[0], errors='coerce')

# Floor_Number: '12', 'Ground', '4th   of 5 Floors', '2 Floors'
fn = df['Floor_Number'].astype('string').str.strip()
floor = pd.to_numeric(fn.where(fn.str.fullmatch(r'\d+')), errors='coerce')
ordinal = fn.str.extract(r'(\d+)(?:st|nd|rd|th)\s+of\s+(\d+)\s+Floors')
floor = floor.fillna(pd.to_numeric(ordinal[0], errors='coerce'))
floor = floor.mask(fn.str.contains('Ground', case=False, na=False), 0)
df['Floor_Number'] = floor
# recover Total_Floors from the 'of N Floors' pattern where it's missing
df['Total_Floors'] = df['Total_Floors'].fillna(pd.to_numeric(ordinal[1], errors='coerce'))

# Car_Parking: sum all numbers in the cell ('1 Covered, 1 Open' -> 2)
df['Car_Parking'] = df['Car_Parking'].astype('string').str.findall(r'\d+').map(
    lambda v: sum(map(int, v)) if isinstance(v, list) and v else np.nan)

# Balconies: numeric only, type words -> missing
df['Balconies'] = pd.to_numeric(df['Balconies'], errors='coerce')

# Furnishing
furn = df['Furnishing'].astype('string').str.strip().str.lower()
df['Furnishing'] = np.select(
    [furn.str.contains('semi', na=False),
     furn.str.fullmatch('unfurnished').fillna(False),
     furn.str.contains('furnish', na=False)],
    ['Semi-Furnished', 'Unfurnished', 'Furnished'], default=None)

# Construction_Status
cs = df['Construction_Status'].astype('string').str.strip().str.lower()
df['Construction_Status'] = np.select(
    [cs.str.contains('ready', na=False),
     cs.str.contains('under construction', na=False),
     cs.str.contains('new launch', na=False)],
    ['Ready to Move', 'Under Construction', 'New Launch'], default=None)

# Facing: valid compass directions only, normalize 'North - East' -> 'North-East'
facing = df['Facing'].astype('string').str.strip().str.replace(r'\s*-\s*', '-', regex=True).str.title()
VALID_FACING = {'East','West','North','South','North-East','North-West','South-East','South-West'}
df['Facing'] = facing.where(facing.isin(VALID_FACING))

# Ownership: real tenure types only ('Ownership Type' is a scraped header, not a value)
own = df['Ownership'].astype('string').str.strip().str.title()
VALID_OWN = {'Freehold','Leasehold','Co-Operative Society','Power Of Attorney','Self Owned'}
df['Ownership'] = own.where(own.isin(VALID_OWN))

# Transaction
tr = df['Transaction'].astype('string').str.strip().str.title()
df['Transaction'] = tr.where(tr.isin({'Resale','New Property','Sale'}))

print('Text columns normalized; rows unchanged:', len(df))

Text columns normalized; rows unchanged: 63076


## Step 4 — Impossible values → missing

Anything physically impossible is set to missing and imputed later: area outside 100–30,000 sqft, more than 60 floors, property age above 100, years after 2026, coordinates outside India.

**Why fixed bounds instead of percentile clipping:** these are data-entry corruption, not outliers. A genuine ₹40 Cr villa should survive; a 111-billion-floor building should not.

In [5]:
def bound(col, lo, hi):
    bad = (df[col] < lo) | (df[col] > hi)
    df.loc[bad, col] = np.nan
    return int(bad.sum())

nulled = {
    'Area_Sqft': bound('Area_Sqft', 100, 30000),
    'BHK': bound('BHK', 1, 12),
    'Bathrooms': bound('Bathrooms', 1, 15),
    'Balconies': bound('Balconies', 0, 10),
    'Car_Parking': bound('Car_Parking', 0, 10),
    'Floor_Number': bound('Floor_Number', 0, 60),
    'Total_Floors': bound('Total_Floors', 1, 60),
    'Property_Age_Years': bound('Property_Age_Years', 0, 100),
    'Year': bound('Year', 2001, 2026),
}
# coordinates outside India -> both to missing
bad_geo = ~df['Latitude'].between(6, 38) | ~df['Longitude'].between(68, 98)
df.loc[bad_geo & df['Latitude'].notna(), ['Latitude', 'Longitude']] = np.nan
nulled['coords'] = int((bad_geo & df['Latitude'].notna()).sum())
print('Impossible values set to missing:', nulled)

Impossible values set to missing: {'Area_Sqft': 325, 'BHK': 129, 'Bathrooms': 145, 'Balconies': 25, 'Car_Parking': 136, 'Floor_Number': 197, 'Total_Floors': 251, 'Property_Age_Years': 407, 'Year': 36, 'coords': 0}


## Step 5 — Contradicting fields: repair + flag

- Floor number **above** the building's total floors → both to missing + flag (can't know which is wrong)
- "New Launch" but 10 years old → trust the age, blank the status + flag
- 1BHK with a 4,000 sqft area → flag only (either value could be the wrong one)

**Why flags instead of dropping:** "this record contradicted itself" is useful information — the model can use it, or you can filter on it. Dropping would cost >13% of the data.

In [6]:
# floor vs total floors
bad_floor = df['Floor_Number'] > df['Total_Floors']
df['Flag_Floor_Impossible'] = bad_floor.fillna(False).astype(int)
df.loc[bad_floor.fillna(False), ['Floor_Number', 'Total_Floors']] = np.nan

# 'new' status but old building -> trust age, blank status
new_status = df['Construction_Status'].isin(['New Launch', 'Under Construction'])
bad_status = new_status & (df['Property_Age_Years'] > 2)
df['Flag_Status_Age_Conflict'] = bad_status.astype(int)
df.loc[bad_status, 'Construction_Status'] = None

# BHK vs area mismatch (per-BHK plausible sqft ranges) -> flag only
size_lo = df['BHK'] * 200      # e.g. 1BHK < 200 sqft is implausible
size_hi = df['BHK'] * 1800 + 1200   # generous upper bound per BHK
df['Flag_Size_Mismatch'] = ((df['Area_Sqft'] < size_lo) | (df['Area_Sqft'] > size_hi)).fillna(False).astype(int)

print('flagged floor-impossible :', int(df['Flag_Floor_Impossible'].sum()))
print('flagged status-age conflict:', int(df['Flag_Status_Age_Conflict'].sum()))
print('flagged size mismatch     :', int(df['Flag_Size_Mismatch'].sum()))

flagged floor-impossible : 134
flagged status-age conflict: 8244
flagged size mismatch     : 2396


## Step 6 — Recompute derived columns

`PricePerSqft`, `Price_Cr`, `Age_Group` and `YearMonth` are recalculated from their source columns, because the scraped versions often disagreed with them (price-per-sqft up to ₹77 Cr).

**Why:** derived columns must never contradict the columns they come from.

In [7]:
df['Price_Cr'] = (df['Price_INR'] / 1e7).round(4)
df['PricePerSqft'] = (df['Price_INR'] / df['Area_Sqft']).round(2)

age_bins = [-0.1, 1, 5, 10, 20, 100]
age_labels = ['New', '1-5 Years', '6-10 Years', '11-20 Years', '20+ Years']
df['Age_Group'] = pd.cut(df['Property_Age_Years'], bins=age_bins, labels=age_labels).astype('string')

df['YearMonth'] = pd.to_datetime(dict(year=df['Year'], month=df['Month'], day=1), errors='coerce').dt.strftime('%Y-%m')
print('Derived columns rebuilt. PricePerSqft now: min %.0f, median %.0f, max %.0f' % (
    df['PricePerSqft'].min(), df['PricePerSqft'].median(), df['PricePerSqft'].max()))

Derived columns rebuilt. PricePerSqft now: min 30, median 8065, max 4090983


## Step 6b — Save the no-imputation variant

Before any value is imputed, save `housedata_clean_without_imputation.csv`: fully cleaned
(duplicates gone, text fixed, impossible values removed, contradictions repaired and flagged,
derived columns rebuilt) but **every remaining gap stays empty** — no guessed values anywhere.

**When to use which output:**
- `housedata_clean.csv` (imputed) — for models that need complete data (linear models, neural nets, sklearn defaults)
- `housedata_clean_without_imputation.csv` — when you want only genuine scraped values; tree models
  (LightGBM, XGBoost, CatBoost) handle missing values natively and often prefer this

In [8]:
no_imp = df.copy()
no_imp['Flag_PPS_Extreme'] = ((no_imp['Price_INR'] / no_imp['Area_Sqft'] > 100000) |
                              (no_imp['Price_INR'] / no_imp['Area_Sqft'] < 500)).fillna(False).astype(int)
feature_cols = [c for c in no_imp.columns if c not in ('URL', 'Full_Address')]
no_imp = no_imp.drop_duplicates(subset=feature_cols).reset_index(drop=True)

# keep pincode formatting consistent with the imputed variant
no_imp['Pincode'] = no_imp['Pincode'].astype('string').str.replace(r'\.0$', '', regex=True)

OUT_NOIMP = 'datasets/housedata_clean_without_imputation.csv'
no_imp.to_csv(OUT_NOIMP, index=False)
missing_cells = int(no_imp[feature_cols].isna().sum().sum())
print('Saved:', OUT_NOIMP)
print(f'{len(no_imp)} rows x {len(no_imp.columns)} columns')
print(f'Missing cells kept as missing: {missing_cells:,} '
      f'({missing_cells / (len(no_imp) * len(feature_cols)) * 100:.1f}% of feature cells)')

Saved: datasets/housedata_clean_without_imputation.csv
62088 rows x 35 columns
Missing cells kept as missing: 356,355 (17.4% of feature cells)


## Step 7 — Fill missing values (MICE)

Three methods were tested by hiding 10% of the known values and measuring reconstruction error (lower = better):

| Method | Error | |
|---|---|---|
| **MICE** | **0.823** | chosen — best on BHK, floors, bathrooms |
| Grouped median | 0.857 | simple, close second |
| KNN (k=5) | 0.918 | worst, and too slow at this scale |

MICE predicts each column from all the others — structural fields explain each other (a 1,650 sqft flat with 3 bathrooms on floor 7 is almost certainly a 3BHK).

**Safeguards:**
- Price is **excluded** from the imputer — using the target to build features is leakage
- Coordinates fill from their locality's centroid first
- Counts are rounded to whole numbers and clipped to valid ranges
- An imputed age may not contradict a known "New Launch" status (capped at 2 yrs)
- `Age_Group` recomputed after imputation; `Pincode` reformatted to clean 6-digit strings
- Categorical gaps become an explicit `'Unknown'` (mode-fill would fake knowledge)

In [9]:
# ---- coordinates first: locality centroid, then global median as last resort
for c in ['Latitude', 'Longitude']:
    df[c] = df[c].astype('float64')
    df[c] = df[c].fillna(df.groupby('Locality')[c].transform('mean'))
    df[c] = df[c].fillna(df[c].median())

# remember which ages are about to be guessed (needed for the status guard below)
age_was_missing = df['Property_Age_Years'].isna()

# ---- MICE over the numeric block (coords + time as helper features, no Price)
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

num_cols = ['Area_Sqft','BHK','Bathrooms','Balconies','Car_Parking','Floor_Number',
            'Total_Floors','Property_Age_Years','Year','Month',
            'Cement_Price_Rs_per_bag_50kg','Steel_TMT_Price_Rs_per_tonne']
mice_matrix = num_cols + ['Latitude', 'Longitude']

df[mice_matrix] = df[mice_matrix].astype('float64')
mice = IterativeImputer(max_iter=10, random_state=0)
df[mice_matrix] = mice.fit_transform(df[mice_matrix])

# round count-like columns back to integers and clip to valid ranges
INT_BOUNDS = {'BHK': (1, 12), 'Bathrooms': (1, 15), 'Balconies': (0, 10),
              'Car_Parking': (0, 10), 'Floor_Number': (0, 60), 'Total_Floors': (1, 60),
              'Property_Age_Years': (0, 100), 'Year': (2001, 2026), 'Month': (1, 12)}
for c, (lo, hi) in INT_BOUNDS.items():
    df[c] = df[c].round().clip(lo, hi)
df['Area_Sqft'] = df['Area_Sqft'].clip(100, 30000).round()
# imputation must not reintroduce the floor > total-floors impossibility
df['Floor_Number'] = df[['Floor_Number', 'Total_Floors']].min(axis=1)

# guard: an IMPUTED age may not contradict a KNOWN 'new' construction status
new_status = df['Construction_Status'].isin(['New Launch', 'Under Construction'])
df.loc[age_was_missing & new_status, 'Property_Age_Years'] = \
    df.loc[age_was_missing & new_status, 'Property_Age_Years'].clip(upper=2)

# Age_Group recomputed AFTER imputation so it always matches the final age
age_bins = [-0.1, 1, 5, 10, 20, 100]
age_labels = ['New', '1-5 Years', '6-10 Years', '11-20 Years', '20+ Years']
df['Age_Group'] = pd.cut(df['Property_Age_Years'], bins=age_bins, labels=age_labels).astype('string')

# Pincode back to clean 6-digit strings (CSV round-trip turns them into '500084.0')
for c in ['Pincode']:
    df[c] = df[c].astype('string').str.replace(r'\.0$', '', regex=True)

# ---- categoricals: explicit Unknown
cat_cols = ['Furnishing','Facing','Ownership','Transaction','Construction_Status',
            'Age_Group','Locality','Society','Overlooking','YearMonth','Pincode','Possible_Pincodes']
for c in cat_cols:
    df[c] = df[c].astype('string').fillna('Unknown').replace('', 'Unknown')

# derived columns after imputation
df['PricePerSqft'] = (df['Price_INR'] / df['Area_Sqft']).round(2)

feature_cols = [c for c in df.columns if c not in ('URL','Full_Address')]
print('Missing values remaining in feature columns:', int(df[feature_cols].isna().sum().sum()))
print('Status-age conflicts after guard:',
      int((df['Construction_Status'].isin(['New Launch','Under Construction'])
           & (df['Property_Age_Years'] > 2)).sum()))

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Missing values remaining in feature columns: 0
Status-age conflicts after guard: 0


## Step 8 — Final guards & save

- `Flag_PPS_Extreme`: price and area that are absurd **together** (per-sqft above ₹1 lakh or below ₹500) — flagged, not dropped
- Final dedupe: imputation can make two near-identical rows fully identical

**For model training:** drop `URL`, `Full_Address` (identifiers) and `PricePerSqft`, `Price_Cr` (computed from the target — leakage). Encode categoricals after the train/test split. The strictest dataset is `all Flag_* == 0` (~83% of rows).

In [10]:
# jointly-absurd price/area pairs -> flag for train-time filtering
df['Flag_PPS_Extreme'] = ((df['PricePerSqft'] > 100000) | (df['PricePerSqft'] < 500)).astype(int)
print('flagged extreme price-per-sqft:', int(df['Flag_PPS_Extreme'].sum()))

# imputation can turn near-duplicates into exact duplicates -> final dedupe
feature_cols = [c for c in df.columns if c not in ('URL', 'Full_Address')]
before = len(df)
df = df.drop_duplicates(subset=feature_cols).reset_index(drop=True)
print(f'Final dedupe removed {before - len(df)} imputation-created duplicates')

OUT = 'datasets/housedata_clean.csv'
df.to_csv(OUT, index=False)

print('\nSaved:', OUT)
print(f'Final: {len(df)} rows x {len(df.columns)} columns')
print(f'Row retention: {len(df)}/{start_rows} = {len(df)/start_rows*100:.1f}%')
print('\nTarget summary (Price_INR):')
print(df['Price_INR'].describe().round(0).to_string())

flagged extreme price-per-sqft: 1289
Final dedupe removed 1029 imputation-created duplicates



Saved: datasets/housedata_clean.csv
Final: 62047 rows x 35 columns
Row retention: 62047/93621 = 66.3%

Target summary (Price_INR):
count    6.204700e+04
mean     2.175866e+07
std      3.307187e+07
min      2.000000e+05
25%      8.000000e+06
50%      1.350000e+07
75%      2.300000e+07
max      2.000000e+09
